# Celery 태스크 테스트 노트북

각 셀은 용도가 나뉘어 있습니다. 필요한 것만 실행하세요.

- **셀 1**: 공통 import
- **셀 2~4**: 워커에 태스크 보내고 결과 확인 (비동기)
- **셀 5**: Eager 모드 — 노트북 커널에서 직접 실행 (디버깅)
- **셀 6**: 서비스 직접 호출 — Celery 우회 (가장 빠른 디버깅)
- **셀 7**: 이전 task_id로 결과 조회

## 1. 공통 import

In [ ]:
# from app.worker.celery_app import celery
# from app.worker.tasks.embedding import embed_fruit_images
print(12345)

ModuleNotFoundError: No module named 'app'

## 2. 워커로 태스크 보내기 (비동기)

`delay()`는 큐에 넣고 즉시 리턴. 실제 실행은 celery 컨테이너에서.

In [ ]:
task = embed_fruit_images.delay()
print(f"task_id: {task.id}")
print(f"status: {task.status}")
task

## 3. 상태 폴링 (여러 번 재실행 가능)

In [ ]:
task.status  # PENDING → STARTED → SUCCESS / FAILURE

In [ ]:
task.ready(), task.result

## 4. 완료 대기 (타임아웃 주의)

첫 모델 다운로드 중이면 600초 이상 걸릴 수 있음. 그럴 땐 셀 3으로 폴링.

In [ ]:
result = task.get(timeout=60)
print(result)

## 5. Eager 모드 — 현재 커널에서 직접 실행 (디버깅용 ⭐)

태스크를 큐에 안 태우고 **노트북 안에서 sync 실행**. 예외 트레이스백도 바로 보이고 `breakpoint()`도 작동.

In [ ]:
celery.conf.task_always_eager = True
celery.conf.task_eager_propagates = True

result = embed_fruit_images.delay()
print(result.get())

In [ ]:
# 워커로 돌리고 싶으면 다시 끄기
celery.conf.task_always_eager = False

## 6. 서비스 직접 호출 — Celery 우회 (가장 빠름)

로직 버그·리턴값·타입 확인엔 이게 제일 빠릅니다. 예: `PosixPath` 직렬화 문제도 이걸로 즉시 드러남.

In [3]:
from app.services.point.fruit import FruitPointService

service = FruitPointService()
points = service.build_points()
# type(points), points
print(points)

ModuleNotFoundError: No module named 'app'

## 7. 이전 task_id로 결과 조회

Redis 백엔드에 남아있는 동안(기본 1일) 언제든 가져올 수 있음.

In [ ]:
from celery.result import AsyncResult

task_id = "여기에-이전-task-id"
r = AsyncResult(task_id, app=celery)
r.status, r.result